# 04 — Latent Potential Modeling

We estimate `Maximum_Monthly_Liters` for **January 2026** via three independent estimators and ensemble them. The three are designed to fail in *different ways* so the blend is robust:

| Method | Strength | Weakness |
|---|---|---|
| Peer ceiling | works for outlets with no unconstrained months; non-parametric | sensitive to clustering quality |
| Quantile regression (τ=0.90) | captures the upper envelope of the conditional distribution | leans on observed maxes, which can themselves be censored |
| Unconstrained extrapolation | most direct empirical estimate when it applies | only fires for outlets with ≥2 unconstrained months |

Final = per-row weight-renormalized blend with sanity floor.

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src import config
from src.utils.io import read_parquet
from src.modeling.latent_potential_model import (
    estimate_peer_ceiling,
    estimate_quantile_regression,
    estimate_unconstrained_extrapolation,
    ensemble_predictions,
    run_modeling,
)

pd.set_option("display.max_columns", 80)

In [ ]:
features = read_parquet(config.GOLD_DIR / "outlet_features.parquet")
season_path = config.SILVER_DIR / "seasonality.parquet"
seasonality = read_parquet(season_path) if season_path.exists() else pd.DataFrame()
print(features.shape, seasonality.shape)
display(features.head(3))

## 1. Method-by-method estimates

We run each estimator independently first so we can inspect coverage and distributional sanity per method before blending.

In [ ]:
peer = estimate_peer_ceiling(features)
qreg = estimate_quantile_regression(features)
unc  = estimate_unconstrained_extrapolation(features, seasonality)

methods = pd.DataFrame({
    "peer_ceiling": peer,
    "quantile_regression": qreg,
    "unconstrained_extrapolation": unc,
})
coverage = methods.notna().mean().to_frame("non_null_share")
display(coverage)
display(methods.describe())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))
for ax, col in zip(axes, methods.columns):
    s = methods[col].replace([np.inf, -np.inf], np.nan).dropna()
    if not s.empty:
        ax.hist(np.log1p(s), bins=40)
        ax.set_title(f"{col}\nn={len(s):,}  median={s.median():.0f}")
        ax.set_xlabel("log1p(potential)")
plt.tight_layout()

## 2. Method agreement

Pairwise correlations between estimators tell us how much each is contributing independent information. Very high correlation ⇒ the blend is essentially a single method; very low ⇒ blending will be unstable.

In [ ]:
corr = methods.corr(method="spearman")
display(corr.style.background_gradient(cmap="RdBu_r", vmin=-1, vmax=1).format("{:.2f}"))

## 3. Ensemble + sanity floor

The blended potential must (a) be non-negative and (b) be at least as large as the outlet's historical max, because we know demand was at least that high.

In [ ]:
blended = ensemble_predictions(peer, qreg, unc)
hist_max = features.get("hist_total_max", pd.Series(0, index=features.index)).fillna(0)
floor_mult = config.MODEL_CONFIG["potential_floor_multiplier"]
final = np.maximum(blended.fillna(hist_max * floor_mult), hist_max * floor_mult)

uplift = (final / hist_max.replace(0, np.nan)).replace([np.inf, -np.inf], np.nan)
print("uplift over historical max:")
print(uplift.describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9, 0.99]))
print(f"\noutlets where potential < hist_max (should be 0): {int((final < hist_max).sum())}")

## 4. Sensitivity analysis

Vary key knobs — peer-cluster count, quantile τ, ensemble weights — and confirm the final prediction is stable. We want most outlets to move <10% under reasonable perturbations.

In [ ]:
rows = []
base = blended
for k in (15, 25, 40):
    p = estimate_peer_ceiling(features, k=k)
    b = ensemble_predictions(p, qreg, unc)
    diff = ((b - base).abs() / base.replace(0, np.nan)).median()
    rows.append({"perturbation": f"peer_k={k}", "median_rel_change": float(diff)})

for tau in (0.85, 0.90, 0.95):
    q = estimate_quantile_regression(features, tau=tau)
    b = ensemble_predictions(peer, q, unc)
    diff = ((b - base).abs() / base.replace(0, np.nan)).median()
    rows.append({"perturbation": f"qreg_tau={tau}", "median_rel_change": float(diff)})

for w_peer in (0.3, 0.4, 0.5):
    w = {"peer_ceiling": w_peer, "quantile_regression": (1-w_peer)/2, "unconstrained_extrapolation": (1-w_peer)/2}
    b = ensemble_predictions(peer, qreg, unc, weights=w)
    diff = ((b - base).abs() / base.replace(0, np.nan)).median()
    rows.append({"perturbation": f"w_peer={w_peer}", "median_rel_change": float(diff)})

display(pd.DataFrame(rows))

## 5. Write the final deliverable CSV

`run_modeling()` does the full pipeline end-to-end and writes the submission CSV.

In [ ]:
result = run_modeling(team_name="galaxy_pegasus")
print(result)

submission = pd.read_csv(result["predictions_csv"])
print(submission.shape)
display(submission.head())
print(submission["Maximum_Monthly_Liters"].describe(percentiles=[0.1, 0.5, 0.9, 0.99]))